In [2]:
import json
import argparse
from enum import Enum
from pydantic import BaseModel
from typing import List, Dict, Any, Optional
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import litellm

from siela.update_agent_policy import update_agent_policy
from siela.generate_tool_prompt import generate_tool_prompt
from siela.utils import restore_agent_policy, delete_tool_prompts


import litellm

res = litellm.completion(
    model = 'claude-3-7-sonnet',
    messages = [
        {"role": "user", "content": "What is the capital of France?"},
    ],
    custom_llm_provider = "openai",
)

print(res.choices[0].message['content'])

The capital of France is Paris. Paris is not only the capital but also the largest city in France, serving as the country's major cultural, economic, and political center.


**Exp. 1 Telecom Domain Policy Improvement and Tool Prompt Generation** <br>

In [33]:
error_analysis_file = "err_ana_results/GT_telecom_baseline_llm_agent_claude-3-7-sonnet.json"
domain = 'telecom'
if not Path(error_analysis_file).exists():
    print(f"Error analysis file {error_analysis_file} does not exist.")

In [34]:
assert domain == 'telecom', "This script is only for telecom domain."
# restore the agent policy before running the tool prompt generation
restore_agent_policy(
    domain=domain,
)
# delete existing tool prompts
# delete_tool_prompts(domain=domain)

Restored telecom policies to /home/lijiah/workspace/tau2-bench/data/tau2/domains/telecom/main_policy.md and /home/lijiah/workspace/tau2-bench/data/tau2/domains/telecom/tech_support_manual.md.


In [19]:
updated_policy = update_agent_policy(
    error_analysis_file=error_analysis_file,
    domain=domain,
    model_name='gpt-4.1',
)
for p in updated_policy:
    print(p)

Agent policy loaded for telecom domain.


# Telecom Agent Policy

The current time is 2025-02-25 12:08:00 EST.

As a telecom agent, you can help users with  **technical support**, **overdue bill payment**, **line suspension**, and **plan options**.

You should not provide any information, knowledge, or procedures not provided by the user or available tools, or give subjective recommendations or comments.

You should only make one tool call at a time, and if you make a tool call, you should not respond to the user simultaneously. If you respond to the user, you should not make a tool call at the same time.

You should deny user requests that are against this policy.

You should transfer the user to a human agent if and only if the request cannot be handled within the scope of your actions. To transfer, first make a tool call to transfer_to_human_agents, and then send the message 'YOU ARE BEING TRANSFERRED TO A HUMAN AGENT. PLEASE HOLD ON.' to the user.

**Before transferring to a human agent, you must try all available and rele

In [ ]:
## Update tool prompts
# Assert that the domain is present in the error analysis file
assert domain in error_analysis_file, f"Domain {domain} not found in {error_analysis_file}"
tool_prompt = generate_tool_prompt(
    error_file_path=error_analysis_file,
    domain=domain,
    model_name='gemini-2.5-pro'
)

for name, tool in tool_prompt['agent_tools'].items():
    print(name)
for name, tool in tool_prompt['user_tools'].items():
    print(name)

Saved 6 agent tool and 0 user tool prompts for 'telecom'
enable_roaming
transfer_to_human_agents
toggle_roaming
refuel_data
reset_apn_settings
grant_app_permission


In [4]:
for name, tool in tool_prompt['user_tools'].items():
    print(name)

toggle_roaming


In [7]:
for name, tool in tool_prompt['agent_tools'].items():
    print(tool)
for name, tool in tool_prompt['user_tools'].items():
    print(tool)

## When to call the tool
- Use this tool when a user with mobile data or MMS issues is traveling abroad, or when you have confirmed their roaming is disabled on the account side.

## Before calling the tool
- **Required Check:** First, use the `get_details_by_id` tool with the user's `line_id`. Check the output for the `"roaming_enabled": false` status.
- **User Interaction:** If the user has not mentioned they are abroad, but you see roaming is disabled, ask for their current location (e.g., "Are you currently traveling outside your home country?") to confirm a roaming issue is relevant.
- **Required Parameters:** You must provide the correct `customer_id` and `line_id`.

## After calling the tool
- **CRITICAL NEXT STEP:** This tool only enables roaming on the network account. It does NOT enable it on the user's physical device. You MUST immediately guide the user to enable 'Data Roaming' in their phone's settings. You can instruct them by saying, "I've enabled roaming on your account

**Exp. 2 Airline Domain Policy Improvement and Tool Prompt Generation** <br>

In [36]:
error_analysis_file = "err_ana_results/GT_airline_baseline_llm_agent_claude-3-7-sonnet.json"
domain = 'airline'
if not Path(error_analysis_file).exists():
    print(f"Error analysis file {error_analysis_file} does not exist.")


In [ ]:
assert domain == 'airline', "This script is only for airline domain."

# Restore the agent policy for the domain
restore_agent_policy(
    domain=domain,
)

# Delete existing tool prompts for the domain
# delete_tool_prompts(domain=domain)





In [25]:

# Assert that the domain is present in the error analysis file
assert domain in error_analysis_file, f"Domain {domain} not found in {error_analysis_file}"

# Update agent policy based on the error analysis file
updated_policy = update_agent_policy(
    error_analysis_file=error_analysis_file,
    domain=domain,
    model_name='gpt-4.1',
)

for p in updated_policy:
    print(p)


Agent policy loaded for airline domain.
# Airline Agent Policy

**Current Time:** 2024-05-15 15:00:00 EST

As an airline agent, you are responsible for assisting users with **booking**, **modifying**, or **canceling** flight reservations, as well as handling **refunds and compensation**. Always strictly follow the procedures and constraints outlined below.

---

## General Operational Guidelines

- **User Confirmation:** Before any action that will update the booking database (including booking, modifying flights, editing baggage, changing cabin class, or updating passenger information), you must:
    1. Clearly present the full action details (including all financial implications) to the user, 
    2. Explicitly ask for the user's confirmation (request a 'yes' or clear affirmative response),
    3. Only proceed to make the database-updating tool call after user confirmation has been received.

    *Example:*
    ```
    The total charge for booking flights X and Y for 3 passengers in 

In [31]:
# Update tool prompts

tool_prompt = generate_tool_prompt(
    error_file_path=error_analysis_file,
    domain=domain,
    model_name='gemini-2.5-pro',
)
for name, tool in tool_prompt['agent_tools'].items():
    print(name)

for name, tool in tool_prompt['user_tools'].items():
    print(name)

Saved 7 agent tool and 0 user tool prompts for 'airline'
cancel_reservation
update_reservation_flights
book_reservation
search_direct_flight
search_onestop_flight
send_certificate
transfer_to_human_agents


In [12]:
for name, tool in tool_prompt['agent_tools'].items():
    print(tool)

for name, tool in tool_prompt['user_tools'].items():
    print(name)

Tool Prompt for cancel_reservation

## When to call the tool
- Use this tool only when a user requests to cancel a reservation, and the reservation meets the eligibility requirements set by policy. These typically include one of the following: (1) Reservation was booked within 24 hours; (2) Reservation was cancelled by the airline; (3) Reservation is a business class flight; (4) Reservation has insurance for a covered reason (e.g. health or weather); (5) Other policy-specified cases. The API does not enforce these rules, so you must check eligibility manually using reservation details.

## Before calling the tool
Required Parameter:
- `reservation_id`: The unique reservation ID for the reservation to be canceled.

Pre-validation Steps:
1. Retrieve reservation details using the `get_reservation_details` tool to verify eligibility. Check the booking date (`created_at`), flight class, insurance status, and user-provided reason.
2. Explicitly confirm all eligibility rules are met per polic

**Exp. 3 Retail Domain Policy Improvement and Tool Prompt Generation** <br>

In [13]:
error_analysis_file = "err_ana_results/GT_retail_baseline_llm_agent_gpt-4.1.json"
domain = 'retail'
if not Path(error_analysis_file).exists():
    print(f"Error analysis file {error_analysis_file} does not exist.")

In [16]:
assert domain == 'retail', "This script is only for retail domain."

In [14]:
# Assert that the domain is present in the error analysis file
assert domain in error_analysis_file, f"Domain {domain} not found in {error_analysis_file}"

# Update agent policy based on the error analysis file
updated_policy = update_agent_policy(
    error_analysis_file=error_analysis_file,
    domain=domain,
    model_name='gpt-4.1',
)

for p in updated_policy:
    print(p)

Agent policy loaded for retail domain.


# Retail Agent Policy (Improved)

As a retail agent, you can help users perform the following actions:

- **Cancel or modify pending orders**
- **Return or exchange delivered orders**
- **Modify their default user address**
- **Provide information about their own profile, orders, and related products**

---

## I. User Authentication

- Begin each conversation by authenticating the user’s identity.
    - Locate the user id via **email**, or via **name + zip code**, even if the user already provides the user id.
    - Do NOT proceed to any other tasks before completing authentication.

- Serve only **one user per conversation**. If a user requests help with another user's account or orders, **deny** the request.

---

## II. General Protocol and Constraints

- **One Tool Call Per Turn:**  
    - At any point, you must perform at most **one tool call per turn**.  
    - Do NOT make multiple tool calls in a single turn.  
    - If multiple actions are required, perform them **sequentially

In [15]:
tool_prompt = generate_tool_prompt(
    error_file_path=error_analysis_file,
    domain=domain,
    model_name='gpt-4.1',
)
for name, tool in tool_prompt['agent_tools'].items():
    print(name)

for name, tool in tool_prompt['user_tools'].items():
    print(name)

Saved 7 agent tool and 0 user tool prompts for 'retail'
exchange_delivered_order_items
modify_pending_order_items
cancel_pending_order
modify_pending_order_address
return_delivered_order_items
get_product_details
calculate
